# LLM Fine-Tuning Cookbook: LoRA & QLoRA



![Python](https://img.shields.io/badge/Python-3.10%2B-blue)

![PyTorch](https://img.shields.io/badge/PyTorch-2.x-ee4c2c)

![Transformers](https://img.shields.io/badge/Transformers-4.40%2B-orange)

![PEFT](https://img.shields.io/badge/PEFT-0.11%2B-green)

![License](https://img.shields.io/badge/License-MIT-lightgrey)



---

## TL;DR



Fine-tuning large language models used to require **hundreds of GBs** of VRAM.

With **LoRA** (Low-Rank Adaptation) and **QLoRA** (Quantized LoRA) you can

fine-tune a 7 B-parameter model on a **single 16 GB GPU** while retaining

> 97 % of full fine-tuning quality.



This notebook walks through **everything** from first principles to a

production-ready training pipeline you can submit to Kaggle competitions.

## Table of Contents



1. [Setup & Imports](#1-setup--imports)

2. [Understanding LLM Architecture](#2-understanding-llm-architecture)

3. [Full Fine-Tuning vs Parameter-Efficient](#3-full-fine-tuning-vs-parameter-efficient)

4. [LoRA Deep Dive](#4-lora-deep-dive)

5. [QLoRA Explained](#5-qlora-explained)

6. [Practical Fine-Tuning Pipeline](#6-practical-fine-tuning-pipeline)

7. [Advanced Techniques](#7-advanced-techniques)

8. [Evaluation & Inference](#8-evaluation--inference)

9. [Deployment](#9-deployment)

10. [Key Takeaways & Further Reading](#10-key-takeaways--further-reading)

---

## 1. Setup & Imports



We install and import the core libraries.

On Kaggle the GPU runtime already has most of these, but we pin versions

for reproducibility.

In [ ]:
# ── Install Kaggle-missing deps (trl ships outside the base image) ──

!pip install -q 'trl<0.11' bitsandbytes  # pin: notebook uses the classic SFTTrainer API


import os

import random

import numpy as np

import torch

import torch.nn as nn

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel

from datasets import load_dataset

from trl import SFTTrainer


SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)

random.seed(SEED)

np.random.seed(SEED)

torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

    torch.backends.cudnn.deterministic = True

    torch.backends.cudnn.benchmark = False


print(f"PyTorch  : {torch.__version__}")

print(f"CUDA     : {torch.cuda.is_available()}")

print(f"Seed     : {SEED}")

if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")

    print(f"VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

> **Key Takeaway -- Setup**

>

> The stack is **torch + transformers + peft + bitsandbytes + trl**.

> `trl.SFTTrainer` wraps HuggingFace `Trainer` with LoRA/QLoRA best practices

> baked in.

---

## 2. Understanding LLM Architecture



Before fine-tuning, we need to know **what** we are tuning.

A decoder-only transformer (GPT-style) stacks *N* identical blocks, each

containing a multi-head self-attention layer and a feed-forward network (FFN).

In [ ]:
def count_parameters(model):
    """Return total and trainable parameter counts."""

    total = sum(p.numel() for p in model.parameters())

    train = sum(p.numel() for p in model.parameters() if p.requires_grad)

    return total, train


# Example: a small GPT-2 style model

from transformers import GPT2LMHeadModel, GPT2Config


cfg = GPT2Config(n_layer=12, n_head=12, n_embd=768)

demo_model = GPT2LMHeadModel(cfg)

total, trainable = count_parameters(demo_model)

print(f"Total params     : {total / 1e6:.1f} M")

print(f"Trainable params : {trainable / 1e6:.1f} M")

In [ ]:
def estimate_memory_gb(num_params, dtype_bytes=2, optimizer_factor=2):
    """Rough GPU memory estimate for training.



    Parameters

    ----------

    num_params : int

    dtype_bytes : int  (2 = fp16, 4 = fp32)

    optimizer_factor : int  (2 for AdamW states: m + v)

    """

    model_mem = num_params * dtype_bytes

    grad_mem = num_params * dtype_bytes

    optim_mem = num_params * 4 * optimizer_factor  # optimizer states in fp32

    total_bytes = model_mem + grad_mem + optim_mem

    return total_bytes / (1024**3)


for size_b in [1, 3, 7, 13, 70]:
    mem = estimate_memory_gb(size_b * 1e9)

    print(f"{size_b:>3d}B params -> ~{mem:6.1f} GB (fp16 + AdamW)")

> **Key Takeaway -- Architecture**

>

> A 7 B model needs **~112 GB** just for weights + gradients + optimizer.

> That is far beyond a single consumer GPU.  We need a smarter approach.

---

## 3. Full Fine-Tuning vs Parameter-Efficient



| Aspect | Full Fine-Tuning | LoRA / QLoRA |

|--------|------------------|--------------|

| Trainable params | 100 % | 0.1 -- 2 % |

| GPU VRAM (7 B) | ~112 GB | ~6 -- 16 GB |

| Training speed | Baseline | 1.5 -- 3x faster |

| Risk of catastrophic forgetting | Higher | Lower |

| Checkpoint size | Full model | Adapter only (few MB) |

In [ ]:
import pandas as pd

import matplotlib.pyplot as plt


rows = []

for size_b in [1, 3, 7, 13]:
    n = size_b * 1e9

    full_mem = estimate_memory_gb(n)

    # LoRA: only ~1% params are trainable but we still load the model in fp16

    lora_mem = (n * 2) / (1024**3) + estimate_memory_gb(n * 0.01)

    # QLoRA: model in 4-bit, trainable adapters in fp16

    qlora_mem = (n * 0.5) / (1024**3) + estimate_memory_gb(n * 0.01)

    rows.append(
        {
            "Model": f"{size_b}B",
            "Full FT (GB)": f"{full_mem:.1f}",
            "LoRA (GB)": f"{lora_mem:.1f}",
            "QLoRA (GB)": f"{qlora_mem:.1f}",
        }
    )


df = pd.DataFrame(rows)

print(df.to_string(index=False))


# Convert formatted strings back to numeric for charting

plot_df = df.copy()

for col in ["Full FT (GB)", "LoRA (GB)", "QLoRA (GB)"]:
    plot_df[col] = plot_df[col].astype(float)


ax = plot_df.set_index("Model")[["Full FT (GB)", "LoRA (GB)", "QLoRA (GB)"]].plot(
    kind="bar",
    figsize=(10, 5),
)

ax.set_title("VRAM Comparison: Full FT vs LoRA vs QLoRA")

ax.set_ylabel("Estimated Memory (GB)")

ax.grid(axis="y", alpha=0.25)

plt.tight_layout()

plt.show()

> **Key Takeaway -- Efficiency**

>

> QLoRA lets you fine-tune a **7 B model in ~6 GB VRAM** -- that fits on a

> free Kaggle T4 GPU!

### Insight: Memory vs Quality Trade-off



- **Observation:** the memory gap between full fine-tuning and QLoRA grows as model size increases.

- **Because** QLoRA keeps the frozen base in 4-bit, large models remain trainable on commodity GPUs.

- **Therefore**, you can iterate faster on prompts, data curation, and evaluation instead of waiting on infra.

- **Limitation:** extremely low-rank adapters may underfit niche domain style unless you tune rank and alpha.

---

## 4. LoRA Deep Dive



### The Math



Instead of updating the full weight matrix $W \in \mathbb{R}^{d \times d}$,

LoRA decomposes the update into two low-rank matrices:



$$W' = W + \Delta W = W + BA$$



where $B \in \mathbb{R}^{d \times r}$ and $A \in \mathbb{R}^{r \times d}$,

with rank $r \ll d$.



**Parameter savings:** $d^2 \to 2dr$. For $d = 4096, r = 16$ this is

$16.7\text{M} \to 131\text{K}$ -- a **128x** reduction.

In [ ]:
class LoRALayer(nn.Module):
    """Minimal LoRA layer implemented from scratch."""

    def __init__(self, in_features, out_features, rank=8, alpha=16):
        super().__init__()

        self.linear = nn.Linear(in_features, out_features, bias=False)

        self.linear.weight.requires_grad = False  # freeze original

        # Low-rank adapter matrices

        self.lora_A = nn.Parameter(torch.randn(rank, in_features) * 0.01)

        self.lora_B = nn.Parameter(torch.zeros(out_features, rank))

        self.scaling = alpha / rank

    def forward(self, x):
        base = self.linear(x)

        # Low-rank path:  x @ A^T @ B^T  (scaled)

        lora = (x @ self.lora_A.T @ self.lora_B.T) * self.scaling

        return base + lora


# Quick sanity check

layer = LoRALayer(768, 768, rank=16, alpha=32)

total, trainable = count_parameters(layer)

print(f"Frozen   : {(total - trainable):,}")

print(f"Trainable: {trainable:,}  ({100 * trainable / total:.2f} %)")


x = torch.randn(2, 10, 768)

y = layer(x)

print(f"Output shape: {y.shape}")

In [ ]:
# Effect of rank on parameter count

d = 4096

print(f"{'Rank':>6s}  {'LoRA Params':>14s}  {'% of Full':>10s}")

print("-" * 35)

for r in [4, 8, 16, 32, 64, 128, 256]:
    lora_params = 2 * d * r

    full_params = d * d

    print(f"{r:>6d}  {lora_params:>14,}  {100 * lora_params / full_params:>9.2f}%")

> **Key Takeaway -- LoRA**

>

> Rank 16 is the sweet spot for most 7 B models -- it gives ~0.2 % trainable

> parameters and matches full FT quality on most benchmarks.

---

## 5. QLoRA Explained



QLoRA adds **4-bit NormalFloat (NF4)** quantization on top of LoRA:



1. Load the base model in 4-bit precision (0.5 bytes / param)

2. Attach LoRA adapters in fp16 / bf16

3. Train only the adapters; back-prop through the quantized base via

   *double quantization* and *paged optimizers*.



This cuts VRAM by an additional **~3x** compared to fp16 LoRA.

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",  # NormalFloat 4-bit
    bnb_4bit_compute_dtype=torch.bfloat16,  # compute in bf16
    bnb_4bit_use_double_quant=True,  # double quantization
)


print("BitsAndBytesConfig ready:")

print(f"  load_in_4bit        = {bnb_config.load_in_4bit}")

print(f"  quant_type          = {bnb_config.bnb_4bit_quant_type}")

print(f"  compute_dtype       = {bnb_config.bnb_4bit_compute_dtype}")

print(f"  double_quant        = {bnb_config.bnb_4bit_use_double_quant}")

> **Key Takeaway -- QLoRA**

>

> The magic is `load_in_4bit=True` + `bnb_4bit_quant_type="nf4"` +

> `bnb_4bit_use_double_quant=True`. Three flags, 3x VRAM savings.

---

## 6. Practical Fine-Tuning Pipeline



Let's put it all together with a **real** model and dataset.

We'll fine-tune a small model as a demonstration -- swap in any HF model ID

for your competition.

In [ ]:
MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # swap for your target model


tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"


# Keep trust_remote_code disabled by default; only enable for vetted models.

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)

model = prepare_model_for_kbit_training(model)


total, trainable = count_parameters(model)

print(f"Base model loaded  : {total / 1e6:.1f} M params")

print(f"Trainable (before) : {trainable / 1e6:.1f} M params")

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)


model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

In [ ]:
# Using a small instruction-tuning dataset for demonstration

dataset = load_dataset("yahma/alpaca-cleaned", split="train[:2000]")


def format_instruction(sample):
    if sample["input"]:
        text = (
            f"### Instruction:\n{sample['instruction']}\n\n"
            f"### Input:\n{sample['input']}\n\n"
            f"### Response:\n{sample['output']}"
        )

    else:
        text = (
            f"### Instruction:\n{sample['instruction']}\n\n"
            f"### Response:\n{sample['output']}"
        )

    return {"text": text}


dataset = dataset.map(format_instruction)

print(f"Dataset size: {len(dataset)}")

print(f"Sample:\n{dataset[0]['text'][:300]}...")

In [ ]:
training_args = TrainingArguments(
    output_dir="./lora-checkpoints",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    logging_steps=10,
    save_strategy="steps",
    save_steps=50,
    bf16=True,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    max_grad_norm=0.3,
    report_to="none",
)


trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
    tokenizer=tokenizer,
    max_seq_length=512,
    dataset_text_field="text",
    packing=False,
)


print("Trainer ready. Call trainer.train() to start.")

In [ ]:
# Uncomment to actually train (takes ~15-20 min on T4):

# trainer.train()

# trainer.save_model("./lora-final")

print("Training cell ready -- uncomment to run.")

> **Key Takeaway -- Pipeline**

>

> The full recipe is: `BitsAndBytesConfig` -> `AutoModelForCausalLM` ->

> `prepare_model_for_kbit_training` -> `LoraConfig` -> `get_peft_model` ->

> `SFTTrainer`. Six steps.

---

## 7. Advanced Techniques



Push quality even further with these proven tricks.

In [ ]:
# ── Gradient Checkpointing ─────────────────────────────────────────

# Already enabled via TrainingArguments above.  Trades ~30% speed

# for ~60% less activation memory.


# ── Flash Attention 2 ──────────────────────────────────────────────

# Supported on Ampere+ GPUs (A100, H100, RTX 30xx/40xx).

# Just add attn_implementation when loading:

#

# model = AutoModelForCausalLM.from_pretrained(

#     MODEL_ID,

#     quantization_config=bnb_config,

#     device_map="auto",

#     attn_implementation="flash_attention_2",

# )


print("Flash Attention 2: add attn_implementation='flash_attention_2'")

In [ ]:
# ── NEFTune (Noisy Embeddings) ─────────────────────────────────────

# Adds uniform noise to embedding vectors during training.

# Shown to improve chat-style fine-tuning by ~2-5 pts on MT-Bench.

#

# trainer = SFTTrainer(

#     ...,

#     neftune_noise_alpha=5,   # recommended: 5-15

# )


# ── Learning Rate Schedule Comparison ─────────────────────────────

import math


steps = 500

warmup = int(steps * 0.03)

lr_max = 2e-4


def cosine_lr(step):
    if step < warmup:
        return lr_max * step / warmup

    progress = (step - warmup) / (steps - warmup)

    return lr_max * 0.5 * (1 + math.cos(math.pi * progress))


def linear_lr(step):
    if step < warmup:
        return lr_max * step / warmup

    return lr_max * (1 - (step - warmup) / (steps - warmup))


# Print a few sample values

print(f"{'Step':>6s}  {'Cosine':>10s}  {'Linear':>10s}")

for s in [0, 15, 50, 100, 250, 400, 499]:
    print(f"{s:>6d}  {cosine_lr(s):>10.6f}  {linear_lr(s):>10.6f}")

> **Key Takeaway -- Advanced**

>

> Enable **gradient checkpointing** (free VRAM win), try **Flash Attention 2**

> on Ampere GPUs, and experiment with **NEFTune** for chat models.

---

## 8. Evaluation & Inference



After training we need to:

1. **Merge** LoRA weights back into the base model

2. **Generate** text to sanity-check

3. **Benchmark** perplexity on a held-out set

In [ ]:
# ── Merge LoRA into base model ─────────────────────────────────────

# After training, you can merge for faster inference:

#

# from peft import PeftModel

#

# base = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16)

# merged = PeftModel.from_pretrained(base, "./lora-final")

# merged = merged.merge_and_unload()

# merged.save_pretrained("./merged-model")

# tokenizer.save_pretrained("./merged-model")


print("Merge recipe: PeftModel.from_pretrained -> merge_and_unload -> save")

In [ ]:
# ── Quick generation test ─────────────────────────────────────────

from transformers import pipeline


# For demo we use the (un-trained) model already in memory

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=128,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
)


prompt = "### Instruction:\nExplain LoRA in one sentence.\n\n### Response:\n"

result = pipe(prompt)

print(result[0]["generated_text"])

In [ ]:
# ── Perplexity on a held-out split ───────────────────────────────

import math


def compute_perplexity(model, tokenizer, texts, max_length=512):
    """Compute perplexity over a list of texts."""

    model.eval()

    total_loss = 0.0

    total_tokens = 0

    with torch.no_grad():
        for text in texts:
            enc = tokenizer(
                text, return_tensors="pt", truncation=True, max_length=max_length
            ).to(model.device)

            outputs = model(**enc, labels=enc["input_ids"])

            total_loss += outputs.loss.item() * enc["input_ids"].size(1)

            total_tokens += enc["input_ids"].size(1)

    avg_loss = total_loss / total_tokens

    return math.exp(avg_loss)


# Usage (uncomment when model is trained):

# eval_texts = load_dataset("yahma/alpaca-cleaned", split="train[2000:2100]")["output"]

# ppl = compute_perplexity(model, tokenizer, eval_texts)

# print(f"Perplexity: {ppl:.2f}")


print("Perplexity benchmark ready -- uncomment after training.")

> **Key Takeaway -- Evaluation**

>

> Always merge before deployment (`merge_and_unload`) and measure

> **perplexity** as a quick sanity metric alongside task-specific evals.

---

## 9. Deployment



Once trained and merged, you have several deployment paths.

In [ ]:
# ── GGUF Export (for llama.cpp / Ollama) ──────────────────────────

# 1. Install llama.cpp:  git clone https://github.com/ggerganov/llama.cpp

# 2. Convert:

#    python llama.cpp/convert_hf_to_gguf.py ./merged-model \\

#        --outfile model.gguf --outtype q4_k_m

# 3. Quantize further (optional):

#    ./llama.cpp/build/bin/llama-quantize model.gguf model-q4.gguf Q4_K_M


print("GGUF export: convert_hf_to_gguf.py -> quantize -> serve with Ollama")

In [ ]:
# ── vLLM Serving ─────────────────────────────────────────────────

# vLLM gives you OpenAI-compatible API with continuous batching.

#

# pip install vllm

# python -m vllm.entrypoints.openai.api_server \\

#     --model ./merged-model \\

#     --dtype auto \\

#     --max-model-len 4096 \\

#     --gpu-memory-utilization 0.90 \\

#     --port 8000

#

# Then query:

# curl http://localhost:8000/v1/completions \\

#   -H 'Content-Type: application/json' \\

#   -d '{"model": "./merged-model", "prompt": "Hello!", "max_tokens": 64}'


print("vLLM: OpenAI-compatible API with PagedAttention & continuous batching")

---

## 10. Key Takeaways & Further Reading



### Summary



| What | Why |

|------|-----|

| **LoRA** | Train <2 % of params, keep >97 % quality |

| **QLoRA** | Add 4-bit quantization for 3x more VRAM savings |

| **Rank 16** | Sweet spot for 7 B models |

| **Cosine LR + warmup** | Stable convergence |

| **Gradient checkpointing** | Free VRAM, slight speed cost |

| **NEFTune** | +2-5 pts on chat benchmarks |

| **Merge + GGUF** | Ship anywhere |

### Kaggle Competition References



These competitions benefit directly from LoRA / QLoRA fine-tuning:



- **Med-Gemma** -- Medical question answering with Google's Med-Gemma.

  Fine-tune with domain-specific medical QA pairs using QLoRA on a T4.

- **Akkadian Translation** -- Translate cuneiform tablets. Low-resource

  language tasks are *ideal* for LoRA because the base model already

  understands language structure; you just teach it a new mapping.

- **AIMO 3 (AI Math Olympiad)** -- Mathematical reasoning. Fine-tune on

  chain-of-thought math traces to boost step-by-step problem solving.

### Further Reading



- [LoRA: Low-Rank Adaptation of Large Language Models](https://arxiv.org/abs/2106.09685) (Hu et al., 2021)

- [QLoRA: Efficient Finetuning of Quantized LLMs](https://arxiv.org/abs/2305.14314) (Dettmers et al., 2023)

- [NEFTune: Noisy Embeddings Improve Instruction Finetuning](https://arxiv.org/abs/2310.05914) (Jain et al., 2023)

- [HuggingFace PEFT Documentation](https://huggingface.co/docs/peft)

- [TRL -- Transformer Reinforcement Learning](https://huggingface.co/docs/trl)

- [bitsandbytes](https://github.com/TimDettmers/bitsandbytes)

- [vLLM](https://docs.vllm.ai/)

---



**Ready to fine-tune?** Fork this notebook, pick a competition model,

swap in `MODEL_ID`, point to your dataset, and hit **Run All**.



If this notebook helped you, please **upvote** and leave a comment!



Happy fine-tuning!